### # using ctas **read_files csv and parquet**
The code in the next cell creates a table using CTAS with the read_files() function.

The read_files() table-valued function (TVF) enables reading a variety of file formats and provides additional options for data ingestion.

In [0]:
%sql
select * from read_files('/Volumes/workspace/default/usr_parquet', format=> 'parquet') limit 10;

In [0]:
%python
df = spark.read.format("parquet").options(header='true', inferSchema='true').load('/Volumes/workspace/default/usr_parquet')

In [0]:
%sql
create table users_info as select * from read_files('/Volumes/workspace/default/usr_parquet', format=> 'parquet');

In [0]:
%sql
select * from users_info;

In [0]:
df.display(10)

In [0]:
%sql
select * from read_files('/Volumes/workspace/default/inbound', format => 'csv') limit 10;

### STREAMING TABLES/AUTOLOADER (SCHEDULE REFRESH)


In [0]:
%sql
create streaming table orders_brz as select * from stream read_files('/Volumes/workspace/default/inbound', format => 'csv');

In [0]:
%sql
DESCRIBE TABLE extended orders_brz;

In [0]:
%sql
describe history orders_brz;

In [0]:
%sql
select count(*) from orders_brz;

In [0]:
%sql
REFRESH STREAMING TABLE orders_brz FULL;

SELECT count(*) FROM orders_brz;

In [0]:
%sql
select * from orders_brz limit 20;

In [0]:
%sql
-- Drop the table if it exists for demonstration purposes
DROP TABLE IF EXISTS sales_bronze;


-- Create the Delta table
CREATE TABLE sales_bronze AS
SELECT 
  *,
  _metadata.file_modification_time AS file_modification_time,
  _metadata.file_name AS source_file, 
  current_timestamp() as ingestion_time 
FROM read_files(
        "/Volumes/workspace/default/inbound/orders.csv",
        format => "csv",
        sep => ",",
        header => true
      );


-- Display the table
SELECT *
FROM sales_bronze

### Ingestion json files handling


In [0]:
%sql
select * from read_files('/Volumes/workspace/default/inbound/000.json',format => 'json');

In [0]:
%sql
Create or replace table json_data_variant 
as select decoded_key,
  offset,
  partition,
  timestamp,
  topic,
  parse_json(decoded_value) as json_variant from json_data;
    


In [0]:
%sql
Select * from json_data_variant;

In [0]:
%sql
select * from json_data;

In [0]:
%sql
select json_variant:device from json_data_variant limit 10;

In [0]:
%sql

create table super_market using delta as select * from read_files('/Volumes/workspace/default/inbound/Superstore_orders.csv', format => 'csv');


In [0]:
%sql
describe table super_market;


In [0]:
%sql
describe history super_market;

## MERGE INTO target t
## USING source s
## ON {merge_condition}
## WHEN MATCHED THEN {matched_action}
## WHEN NOT MATCHED THEN {not_matched_action}


## MERGE WITH SCHEMA EVOLUTION INTO target t
## USING source s
## ON {merge_condition}
## WHEN MATCHED THEN {matched_action}
## WHEN NOT MATCHED THEN {not_matched_action}


In [0]:
%python

df = spark.read.table("workspace.bronze_db.credit_card_raw")
df.display()
df1 = df
